# PAN25 Human to AI

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive')

In [ ]:
from authorship_extractor import AuthorshipMetricsExtractor
import pandas as pd
import numpy as np
import re
from scipy.stats import mannwhitneyu
from tqdm import tqdm

# ============================================================================
# STATISTICAL HELPERS
# ============================================================================

def similarity_percent(mean_a, mean_b):
    """100% means identical; lower means more different."""
    if pd.isna(mean_a) or pd.isna(mean_b): return np.nan
    if mean_a == 0: return 100.0 if mean_b == 0 else 0.0
    diff_pct = abs(mean_b - mean_a) / abs(mean_a) * 100.0
    return 100.0 - min(diff_pct, 100.0)

def sigma_over_mu(values):
    """Coefficient of variation: std/|mean| using sample std (ddof=1)."""
    v = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if len(v) < 2: return np.nan
    mu = np.mean(v)
    if mu == 0: return np.nan
    return float(np.std(v, ddof=1) / abs(mu))

def cohens_d(group1, group2):
    """Cohen's d for independent samples (pooled SD)."""
    x = pd.to_numeric(group1, errors="coerce").dropna().to_numpy(dtype=float)
    y = pd.to_numeric(group2, errors="coerce").dropna().to_numpy(dtype=float)
    n1, n2 = len(x), len(y)
    if n1 < 2 or n2 < 2: return np.nan
    m1, m2 = np.mean(x), np.mean(y)
    s1, s2 = np.std(x, ddof=1), np.std(y, ddof=1)
    pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    if pooled == 0: return np.nan
    return float((m1 - m2) / pooled)

def effect_size_category(d):
    if pd.isna(d): return "N/A"
    ad = abs(d)
    if ad < 0.2: return "Negligible"
    if ad < 0.5: return "Small"
    if ad < 0.8: return "Medium"
    return "Large"

def significance_stars(p):
    if pd.isna(p): return ""
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return ""

# ============================================================================
# METRIC EXTRACTION WITH DYNAMIC BASELINE
# ============================================================================

def extract_metrics_with_baseline(df, text_col="cleaned_body", group_col="source",
                                  baseline_label="human", baseline_sample_size=300,
                                  max_baseline_tokens=3000, mfw_count=100,
                                  keep_cols=("author", "source")):
    """
    Builds a 'Human Norm' baseline, then extracts metrics for all rows.
    """
    print(f"\n--- Building '{baseline_label}' Reference Baseline ---")

    # 1. Filter dataset to strictly the baseline group (e.g., human texts)
    baseline_df = df[df[group_col] == baseline_label]
    if baseline_df.empty:
        raise ValueError(f"No rows found with {group_col} == '{baseline_label}'. Cannot build baseline.")

    # 2. Sample texts to create a macroscopic norm
    n_samples = min(baseline_sample_size, len(baseline_df))
    sample_texts = baseline_df[text_col].dropna().sample(n=n_samples, random_state=42)
    reference_text = " ".join(sample_texts.astype(str).tolist())

    # 3. Tokenize and cap length to keep computation times reasonable
    all_ref_tokens = [t for t in re.findall(r'\b\w+\b', reference_text.lower()) if t.isalpha()]
    reference_tokens = all_ref_tokens[:max_baseline_tokens]
    print(f"Baseline established using {len(reference_tokens)} tokens from {n_samples} documents.")

    # 4. Extract metrics for every document
    rows = []
    total = len(df)
    print(f"\n--- Extracting Metrics for {total} Documents ---")

    for idx, row in tqdm(df.iterrows(), total=total, desc="Processing Documents", unit="doc"):
        text = str(row[text_col])
        extractor = AuthorshipMetricsExtractor(text)

        # Pass the baseline tokens to unlock M13, M14, and M15
        metrics = extractor.extract_all_metrics(text2_tokens=reference_tokens)

        # OVERRIDE M13: Explicitly recalculate Burrows' Delta with your custom MFW count
        metrics['m13_burrows_delta'] = extractor.metric_13_burrows_delta(reference_tokens, mfw_count=mfw_count)

        # Append Metadata
        metrics["doc_id"] = idx
        for c in keep_cols:
            if c in df.columns:
                metrics[c] = row[c]
        rows.append(metrics)

    return pd.DataFrame(rows)

# ============================================================================
# SUMMARY AND STATS GENERATION
# ============================================================================

def summarize_metrics(metrics_df, group_col="source", group_a="human1", group_b="human2",
                      alpha=0.05, bonferroni=True, print_table=True):
    """Calculates means, statistical significance, and effect sizes between two groups."""

    A = metrics_df[metrics_df[group_col] == group_a]
    B = metrics_df[metrics_df[group_col] == group_b]

    # Only grab numeric metric columns starting with 'm'
    metric_cols = sorted([c for c in metrics_df.columns if c.startswith("m") and pd.api.types.is_numeric_dtype(metrics_df[c])])

    rows = []
    for metric in metric_cols:
        a_vals = pd.to_numeric(A[metric], errors="coerce").dropna()
        b_vals = pd.to_numeric(B[metric], errors="coerce").dropna()

        a_mean = a_vals.mean() if len(a_vals) else np.nan
        b_mean = b_vals.mean() if len(b_vals) else np.nan
        diff = (b_mean - a_mean) if (not pd.isna(a_mean) and not pd.isna(b_mean)) else np.nan
        sim = similarity_percent(a_mean, b_mean)

        a_cv = sigma_over_mu(a_vals)
        b_cv = sigma_over_mu(b_vals)
        cv_diff = (b_cv - a_cv) if (not pd.isna(a_cv) and not pd.isna(b_cv)) else np.nan

        d = cohens_d(a_vals, b_vals)
        eff = effect_size_category(d)

        p = np.nan
        try:
            if len(a_vals) and len(b_vals):
                _, p = mannwhitneyu(a_vals, b_vals, alternative="two-sided")
        except Exception:
            pass

        # Dynamically map the columns based on group names
        rows.append({
            "metric": metric,
            f"{group_a}_mean": a_mean,
            f"{group_b}_mean": b_mean,
            f"diff_({group_b}-{group_a})": diff,
            "similarity_pct": sim,
            f"{group_a}_sigma_over_mu": a_cv,
            f"{group_b}_sigma_over_mu": b_cv,
            f"diff_sigma_over_mu_({group_b}-{group_a})": cv_diff,
            "cohens_d": d, "effect_size": eff, "p_value": p, "sig": significance_stars(p),
            f"n_{group_a}": int(len(a_vals)),
            f"n_{group_b}": int(len(b_vals)),
        })

    summary_df = pd.DataFrame(rows)

    if bonferroni:
        m = len(summary_df)
        alpha_adj = alpha / m if m > 0 else alpha
        summary_df["alpha_bonf"] = alpha_adj
        summary_df["sig_bonf"] = summary_df["p_value"].apply(
            lambda pv: "***" if (not pd.isna(pv) and pv < 0.001 and pv <= alpha_adj)
            else ("**" if (not pd.isna(pv) and pv < 0.01 and pv <= alpha_adj)
            else ("*" if (not pd.isna(pv) and pv < 0.05 and pv <= alpha_adj) else ""))
        )

    if print_table:
        pd.set_option("display.float_format", "{:,.6f}".format)
        print("\n" + "=" * 210) # Widened for the extra columns
        print(f"ALL METRICS - {group_a} vs {group_b} (Including σ/μ and Cohen's d)")
        print("=" * 210)

        # Dynamic Headers
        head_a = f"{group_a[:10]} Mean"
        head_b = f"{group_b[:10]} Mean"
        cv_a = f"{group_a[:5]} σ/μ"
        cv_b = f"{group_b[:5]} σ/μ"

        print(f"{'Metric':<35} {head_a:>12} {head_b:>12} {'Diff':>10} {'Sim%':>7} {cv_a:>10} {cv_b:>10} {'Δ σ/μ':>10} {'Cohen d':>9} {'Effect':>12} {'P-value':>12} {'Sig':>5}")
        print("-" * 210)

        for _, r in summary_df.iterrows():
            sim_s = f"{r['similarity_pct']:6.1f}%" if not pd.isna(r["similarity_pct"]) else ""
            print(f"{r['metric']:<35} {r[f'{group_a}_mean']:12.4f} {r[f'{group_b}_mean']:12.4f} {r[f'diff_({group_b}-{group_a})']:10.4f} {sim_s:>7} {r[f'{group_a}_sigma_over_mu']:10.4f} {r[f'{group_b}_sigma_over_mu']:10.4f} {r[f'diff_sigma_over_mu_({group_b}-{group_a})']:10.4f} {r['cohens_d']:9.3f} {r['effect_size']:>12} {r['p_value']:12.6f} {r['sig']:>5}")
        print("=" * 210 + "\n")

    return summary_df

# ============================================================================
# MAIN EXECUTION LOOP
# ============================================================================

if __name__ == "__main__":

    DATA_PATH = "/content/balanced_train_equal_genres.jsonl"
    TEXT_COLUMN = "text"
    GROUP_COLUMN = "label"   # 0 / 1

    df = pd.read_json(DATA_PATH, lines=True)

    # map labels
    df[
        "label_name"] = df["label"].map({0: "Human", 1: "AI"})

    GROUP_COLUMN = "label_name"
    GROUP_A = "Human"
    GROUP_B = "AI"

    BASELINE_LABEL = GROUP_A
    BASELINE_DOC_COUNT = 300
    MAX_BASELINE_TOKENS = 3000
    MFW_COUNT = 100

    # -------- Loop per Genre --------
    genres = df["genre"].unique()

    for genre in genres:
        print(f"\n\n========== PROCESSING GENRE: {genre.upper()} ==========\n")

        genre_df = df[df["genre"] == genre].reset_index(drop=True)

        # ---- Extract metrics ----
        metrics_df = extract_metrics_with_baseline(
            df=genre_df,
            text_col=TEXT_COLUMN,
            group_col=GROUP_COLUMN,
            baseline_label=BASELINE_LABEL,
            baseline_sample_size=BASELINE_DOC_COUNT,
            max_baseline_tokens=MAX_BASELINE_TOKENS,
            mfw_count=MFW_COUNT,
            keep_cols=("model", "label", "genre", "label_name")
        )

        # Save features per genre
        metrics_df.to_csv(f"features_{genre}.csv", index=False)

        # ---- Stats ----
        summary_df = summarize_metrics(
            metrics_df,
            group_col=GROUP_COLUMN,
            group_a=GROUP_A,
            group_b=GROUP_B,
            alpha=0.05,
            bonferroni=True,
            print_table=True
        )

        # Save stats per genre
        summary_df.to_csv(f"stats_summary_new_{genre}.csv", index=False)

        # Paper-ready table
        paper_cols = [
            "metric",
            f"{GROUP_A}_mean",
            f"{GROUP_B}_mean",
            f"diff_({GROUP_B}-{GROUP_A})",
            "similarity_pct",
            f"{GROUP_A}_sigma_over_mu",
            f"{GROUP_B}_sigma_over_mu",
            f"diff_sigma_over_mu_({GROUP_B}-{GROUP_A})",
            "cohens_d",
            "effect_size",
            "p_value",
            "sig_bonf"
        ]

        summary_df[paper_cols].to_csv(f"paper_table_new_{genre}.csv", index=False)

    print("\n Done. Metrics computed per genre.")



========== PROCESSING GENRE: ESSAYS ==========


--- Building 'Human' Reference Baseline ---
Baseline established using 3000 tokens from 300 documents.

--- Extracting Metrics for 1740 Documents ---


Processing Documents: 100%|██████████| 1740/1740 [01:57<00:00, 14.87doc/s]



ALL METRICS - Human vs AI (Including σ/μ and Cohen's d)
Metric                                Human Mean      AI Mean       Diff    Sim%  Human σ/μ     AI σ/μ      Δ σ/μ   Cohen d       Effect      P-value   Sig
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
m01_average_word_length                   4.7901       5.5562     0.7662   84.0%     0.0752     0.0952     0.0200    -1.692        Large     0.000000   ***
m02_type_token_ratio                      0.4883       0.5510     0.0627   87.2%     0.0952     0.1075     0.0123    -1.177        Large     0.000000   ***
m03_yules_k                               0.2467       0.2609     0.0142   94.3%     0.3511     0.4210     0.0699    -0.143   Negligible     0.053357      
m04_hapax_legomena                        0.6969       0.7526     0.0558   92.0%     0.0677     0.0650    -0

Processing Documents: 100%|██████████| 1740/1740 [01:57<00:00, 14.82doc/s]



ALL METRICS - Human vs AI (Including σ/μ and Cohen's d)
Metric                                Human Mean      AI Mean       Diff    Sim%  Human σ/μ     AI σ/μ      Δ σ/μ   Cohen d       Effect      P-value   Sig
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
m01_average_word_length                   4.2320       4.6746     0.4426   89.5%     0.0570     0.0653     0.0083    -1.609        Large     0.000000   ***
m02_type_token_ratio                      0.4732       0.5177     0.0445   90.6%     0.0903     0.1271     0.0368    -0.801        Large     0.000000   ***
m03_yules_k                               0.1490       0.1984     0.0494   66.9%     0.2396     0.4996     0.2600    -0.663       Medium     0.000000   ***
m04_hapax_legomena                        0.7089       0.7653     0.0564   92.0%     0.0723     0.0712    -0

Processing Documents: 100%|██████████| 1740/1740 [01:58<00:00, 14.67doc/s]



ALL METRICS - Human vs AI (Including σ/μ and Cohen's d)
Metric                                Human Mean      AI Mean       Diff    Sim%  Human σ/μ     AI σ/μ      Δ σ/μ   Cohen d       Effect      P-value   Sig
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
m01_average_word_length                   4.7830       5.2220     0.4390   90.8%     0.0601     0.0617     0.0016    -1.438        Large     0.000000   ***
m02_type_token_ratio                      0.4555       0.5035     0.0479   89.5%     0.1581     0.1417    -0.0165    -0.669       Medium     0.000000   ***
m03_yules_k                               0.1618       0.2835     0.1217   24.8%     0.5855     0.3984    -0.1872    -1.167        Large     0.000000   ***
m04_hapax_legomena                        0.6649       0.6992     0.0343   94.8%     0.0964     0.1209     0

In [ ]:
import os

print("Current directory:", os.getcwd())
print("\nFiles here:")
print(os.listdir())

Current directory: /content

Files here:
['.config', 'balanced_train_equal_genres.jsonl', 'features_essays.csv', 'authorship_extractor.py', 'features_fiction.csv', 'stats_summary_new_news.csv', 'paper_table_new_essays.csv', '__pycache__', 'stats_summary_new_fiction.csv', 'stats_summary_new_essays.csv', 'paper_table_new_fiction.csv', 'features_news.csv', 'paper_table_new_news.csv', 'sample_data']


In [ ]:
import zipfile

file_list = [
    "features_fiction.csv", "features_essays.csv", "features_news.csv",
    "stats_summary_new_fiction.csv", "stats_summary_new_essays.csv", "stats_summary_new_news.csv",
    "paper_table_new_fiction.csv", "paper_table_new_essays.csv", "paper_table_new_news.csv"
]

with zipfile.ZipFile("results.zip", "w") as z:
    for file in file_list:
        z.write(file)

from google.colab import files
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>